Title: SEM_application.ipynb

Purpose: 

Author: Onno Nennecke on 19.03.2025 Modified: 21.03.2025

Input data: 

    - This file lies here: 

Output data:

    - This file lies here: 

### Load libraries and functions

In [2]:
# Importing libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import glob
import time
import pandas as pd

# Importing functions
import Functions.wind_model_func as wind_model_func
import Functions.solar_model_func as solar_model_func
import Functions.demand as demand_func
import Functions.grid_func as grid_func
import Functions.config as config

/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


Define variables (all in the config file)


```python
Wind:
- alpha_on = 1/7    # Roughness parameter onshore  
- alpha_off = 0.11  # Roughness parameter offshore  
- ref_height = 10   # Height of wind data  
- v_cutin = 3.5     # Minimum wind speed to start producing power  
- v_cutout = 25     # Maximum wind speed to produce power  
- v_rated = 13      # Wind speed at which the turbine produces maximum power  
- time_oper = 24    # operational time of hub [h/day]  
- Hub Height:
    - # hub_height_on = 80    # Approximation: Onshore 80 m --> Bad Approximation --> Used own grid data instead  
    - # hub_height_off = 120  # Approximation: Offshore 120 m --> Bad Approximation --> Used own grid data instead  

Solar:

- cT_c1 = 4.3       # constant [dC]  
- cT_c2 = 0.943     # constant [-]  
- cT_c3 = 0.028     # constant [dC m2 W-1]  
- cT_c4 = -1.528    # constant [dC s m-1]  
- gamma = -0.005    # constant [--]  
- temp_ref = 25     # reference temperature [dC]  
- gstc = 1000       # standard test conditions [W m-2]  
- shift_doy = 186   # if HadGEM : 180  
```

### Load datasets

In [3]:
# Load installed capacity data
grid_offshore = xr.open_dataset('/climca/people/onennecke/Wind_Solar_MaStR/processed_data/wind_offshore_ic.nc')
grid_offshore = grid_offshore['wind_off_cap']
grid_onshore = xr.open_dataset('/climca/people/onennecke/Wind_Solar_MaStR/processed_data/wind_onshore_ic.nc')
grid_onshore = grid_onshore['wind_on_cap']
grid_solar = xr.open_dataset('/climca/people/onennecke/Wind_Solar_MaStR/processed_data/solar_ic.nc')
grid_solar = grid_solar['solar_cap']

In [4]:
# Load wind height data
grid_onsh_hub_height = xr.open_dataset('/climca/people/onennecke/Wind_Solar_MaStR/processed_data/wind_onshore_height_weighted.nc')
grid_offsh_hub_height = xr.open_dataset('/climca/people/onennecke/Wind_Solar_MaStR/processed_data/wind_offshore_height_weighted.nc')

hub_height_on = grid_onsh_hub_height['wind_on_hub_height']
hub_height_off = grid_offsh_hub_height['wind_off_hub_height']

In [5]:
# Load regridded population weights data
pop_regr_CIESIN_weights = xr.open_dataset('/climca/people/onennecke/population_data/population_regrid_weights.nc')

# Load fit values from vdW Paper
demand_fit_values = xr.open_dataset('/climca/people/onennecke/population_data/demand_fit_values_week.nc')

### Define used models

In [6]:
# Load climate data

MIP = 'ScenarioMIP' # CMIP
Institution = '*'
# ESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'CESM2-WACCM', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-Veg', 'GFDL-CM4', 'GFDL-ESM4', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'MPI-ESM1-2-HR',
#            'MRI-ESM2-0', 'KACE-1-0-G', 'TaiESM1', 'UKESM1-0-LL']
ESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'EC-Earth3', 'GFDL-ESM4', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'KACE-1-0-G', 'TaiESM1', 'UKESM1-0-LL'] # 'EC-Earth3-Veg'
ESMs = ['ACCESS-CM2', 'EC-Earth3', 'MPI-ESM1-2-HR', 'MRI-ESM2-0'] # Datetime Format: datetime64
# ESMs = ['BCC-CSM2-MR', 'CESM2', 'GFDL-ESM4', 'TaiESM1'] # Datetime Format: cftime.DatetimeNoLeap
# ESMs = ['KACE-1-0-G', 'UKESM1-0-LL'] # Datetime Format: cftime.Datetime360Day



# ESMs = ['EC-Earth3'] # 'EC-Earth3-Veg'

# ESMs = ['UKESM1-0-LL']
# ESMs = ['TaiEMS1', 'UKESM1-0LL']
scenario = 'ssp370'
# run = 'r1i1p1f1'
time_res = 'day'
variables = ['sfcWind', 'rsds', 'tas', 'tasmax'] # List of variables
grid_def = '*'
version = '*'


In [7]:
# Check which directories are (already) available and if they contain all the variables
# ESMs = ['EC-Earth3']
for ESM in ESMs:
    path = f'/climca/data/CMIP6/{MIP}/{Institution}/{ESM}/{scenario}/'
    matching_dirs = glob.glob(path)
    print(ESM,': ')
    if len(matching_dirs) >= 1:
        runs = os.listdir(matching_dirs[0])
        print(runs)
        
    else:
        print('No matching directory found')
        continue
    for run in runs:
        run_path = os.path.join(matching_dirs[0], run, 'day')
        print('run_path: ', run_path)
        # Check if all required folders (in `variables`) exist
        # missing_folders = [var for var in variables if not os.path.isdir(os.path.join(run_path, var))]
        
        # if missing_folders:
        #     print(f"Missing folders in {run_path}: {missing_folders}")
        #     continue
        missing_data = [
            var for var in variables 
            if not os.path.isdir(os.path.join(run_path, var)) or 
               not any(
                   os.path.isfile(f) 
                   for f in glob.glob(os.path.join(run_path, var, '*', '*', '*'))  # glob pattern to match files two levels down
               )
        ]
        
        if missing_data:
            print(f"Missing data in {run_path}: {missing_data}")
            continue

ACCESS-CM2 : 
['r4i1p1f1', 'r5i1p1f1', 'r1i1p1f1']
run_path:  /climca/data/CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r4i1p1f1/day
run_path:  /climca/data/CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r5i1p1f1/day
run_path:  /climca/data/CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r1i1p1f1/day
EC-Earth3 : 
['r149i1p1f1', 'r6i1p1f1', 'r4i1p1f1', 'r148i1p1f1', 'r105i1p1f1', 'r9i1p1f1', 'r134i1p1f1', 'r141i1p1f1', 'r146i1p1f1', 'r15i1p1f1', 'r112i1p1f1', 'r117i1p1f1', 'r125i1p1f1', 'r113i1p1f1', 'r106i1p1f1', 'r138i1p1f1', 'r5i1p1f1', 'r137i1p1f1', 'r11i1p1f1', 'r145i1p1f1', 'r114i1p1f1', 'r120i1p1f1', 'r128i1p1f1', 'r135i1p1f1', 'r110i1p1f1', 'r129i1p1f1', 'r132i1p1f1', 'r101i1p1f1', 'r124i1p1f1', 'r127i1p1f1', 'r116i1p1f1', 'r131i1p1f1', 'r121i1p1f1', 'r142i1p1f1', 'r102i1p1f1', 'r133i1p1f1', 'r111i1p1f1', 'r140i1p1f1', 'r136i1p1f1', 'r108i1p1f1', 'r130i1p1f1', 'r150i1p1f1', 'r104i1p1f1', 'r118i1p1f1', 'r109i1p1f1', 'r143i1p1f1', 'r147i1p1f1', 'r126i1p1f1', 'r13i1p1f1', 'r

In [8]:
# Only the loading part of the model for testing 
'''
# List to hold individual datasets (one for each variable)
ds_list = []

for ESM in ESMs:
    print('ESM: ', ESM)
    path = f'/climca/data/CMIP6/ScenarioMIP/*/{ESM}/{scenario}/'
    matching_dirs = glob.glob(path)
    if len(matching_dirs) == 1:
        runs = os.listdir(matching_dirs[0])
    else:
        print(f"Found {len(matching_dirs)} matching directories for {ESM} and {scenario}")
        break
    print('Runs: ', runs)
    for run in runs:
        print('Run: ', run)
        run_path = os.path.join(matching_dirs[0], run, 'day/')
        
        # Check if all required folders (in `variables`) exist
        missing_folders = [var for var in variables if not os.path.isdir(os.path.join(run_path, var))]
        
        if missing_folders:
            print(f"Missing folders in {run_path}: {missing_folders}")
            continue
        for variable in variables:
            print('Variable: ', variable)
            #path =f'/climca/data/CMIP6/CMIP/NCAR/{ESM}/{scenario}/{run}/day/{variable}/gn/v20190308/{variable}_day_{ESM}_{scenario}_{run}_gn_*'
            path = f'/climca/data/CMIP6/ScenarioMIP/*/{ESM}/{scenario}/{run}/day/{variable}/*/*/{variable}_day_{ESM}_{scenario}_{run}_*'
            print('Open: ', path)
            # Open with preprocessing (spatial filtering)
            nc = xr.open_mfdataset(path, preprocess=grid_func.preprocess)
                
        #     # Keep only the desired variable, but retain Dataset structure
            nc = nc[[variable]]
            
            
        #     # Filter to only winter months (October to March)
        #     # nc = nc.sel(time=nc.time.dt.month.isin([10, 11, 12, 1, 2, 3]))

            # Filter to only winter months (October to March)
            nc = nc.sel(time=nc.time.dt.year.isin(range(2015, 2025)))

        #     # Append to list for later merging
            ds_list.append(nc)
            
            ds_list = [ds.drop_vars('height') if 'height' in ds.coords else ds for ds in ds_list]

        # # Combine all into a single dataset
        clim_ds = xr.merge(ds_list)
                
        # if ds_list[2]['tas'].units == 'K':
        #     clim_ds['tas'] = clim_ds['tas'] - 273.15

        # if ds_list[3]['tasmax'].units == 'K':
        #     clim_ds['tasmax'] = clim_ds['tasmax'] - 273.15
            
        # # Regrid the combined dataset
        # combined_ds = grid_func.regrid(clim_ds)
        
        break
'''

'\n# List to hold individual datasets (one for each variable)\nds_list = []\n\nfor ESM in ESMs:\n    print(\'ESM: \', ESM)\n    path = f\'/climca/data/CMIP6/ScenarioMIP/*/{ESM}/{scenario}/\'\n    matching_dirs = glob.glob(path)\n    if len(matching_dirs) == 1:\n        runs = os.listdir(matching_dirs[0])\n    else:\n        print(f"Found {len(matching_dirs)} matching directories for {ESM} and {scenario}")\n        break\n    print(\'Runs: \', runs)\n    for run in runs:\n        print(\'Run: \', run)\n        run_path = os.path.join(matching_dirs[0], run, \'day/\')\n\n        # Check if all required folders (in `variables`) exist\n        missing_folders = [var for var in variables if not os.path.isdir(os.path.join(run_path, var))]\n\n        if missing_folders:\n            print(f"Missing folders in {run_path}: {missing_folders}")\n            continue\n        for variable in variables:\n            print(\'Variable: \', variable)\n            #path =f\'/climca/data/CMIP6/CMIP/NCAR/

In [9]:
all_ts_outputs = []
for ESM in ESMs:
    print('ESM: ', ESM)
    path = f'/climca/data/CMIP6/{MIP}/{Institution}/{ESM}/{scenario}/'
    matching_dirs = glob.glob(path)
    if len(matching_dirs) == 1:
        runs = os.listdir(matching_dirs[0])
    elif len(matching_dirs) >= 1:
        runs = os.listdir(matching_dirs[0]) + os.listdir(matching_dirs[1])
    else:
        print(f"Found {len(matching_dirs)} matching directories for {ESM} and {scenario}")
        break
    print('Runs: ', runs)
    for run in runs:
        ds_list = [] # List to hold individual datasets (one for each variable)

        run_time = time.time()
        print('Run: ', run)
        run_path = os.path.join(matching_dirs[0], run, 'day/') # Watch out for last model (then manually change to matching_dirs[1])
        
        # Check if all required folders (in `variables`) exist
        missing_folders = [var for var in variables if not os.path.isdir(os.path.join(run_path, var))]
        
        if missing_folders:
            print(f"Missing folders in {run_path}: {missing_folders}")
            continue
        
        missing_data = [
            var for var in variables 
            if not os.path.isdir(os.path.join(run_path, var)) or 
               not any(
                   os.path.isfile(f) 
                   for f in glob.glob(os.path.join(run_path, var, '*', '*', '*'))  # glob pattern to match files two levels down
               )
        ]
        
        if missing_data:
            print(f"Missing data in {run_path}: {missing_data}")
            continue
        
        for variable in variables:
            # print('Variable: ', variable)
            #path =f'/climca/data/CMIP6/CMIP/NCAR/{ESM}/{scenario}/{run}/day/{variable}/gn/v20190308/{variable}_day_{ESM}_{scenario}_{run}_gn_*'
            path = f'/climca/data/CMIP6/{MIP}/{Institution}/{ESM}/{scenario}/{run}/{time_res}/{variable}/{grid_def}/{version}/{variable}_{time_res}_{ESM}_{scenario}_{run}_*'
            # print('Open: ', path)
            
            # Filter out files with extensions after .nc
            files = [f for f in glob.glob(path) if f.endswith('.nc')]
            
            # Open with preprocessing (spatial filtering)
            if files:
                nc = xr.open_mfdataset(files, preprocess=grid_func.preprocess)
            else:
                print("No valid .nc files found!")
            # nc = xr.open_mfdataset(path, preprocess=grid_func.preprocess)
                
        #     # Keep only the desired variable, but retain Dataset structure
            nc = nc[[variable]]
            
            
            # Filter years
            nc = nc.sel(time=nc.time.dt.year.isin(range(2015, 2025)))

        #     # Append to list for later merging
            ds_list.append(nc)
            
            ds_list = [ds.drop_vars('height') if 'height' in ds.coords else ds for ds in ds_list]

        # # Combine all into a single dataset
        clim_ds = xr.merge(ds_list)
                
        if ds_list[2]['tas'].units == 'K':
            clim_ds['tas'] = clim_ds['tas'] - 273.15
        
        if ds_list[3]['tasmax'].units == 'K':
            clim_ds['tasmax'] = clim_ds['tasmax'] - 273.15
            
        # Regrid the combined dataset
        combined_ds = grid_func.regrid(clim_ds)
        
        timeseries_ds = combined_ds.copy()
        
        ts_output = timeseries_ds.assign_coords(run = run, ESM = ESM, ESM_run = f'{ESM}_{run}')
        all_ts_outputs.append(ts_output)
        
merged_ts = xr.concat(all_ts_outputs, dim='ESM_run')


ESM:  ACCESS-CM2
Runs:  ['r4i1p1f1', 'r5i1p1f1', 'r1i1p1f1']
Run:  r4i1p1f1
Run:  r5i1p1f1
Run:  r1i1p1f1
ESM:  EC-Earth3
Runs:  ['r149i1p1f1', 'r6i1p1f1', 'r4i1p1f1', 'r148i1p1f1', 'r105i1p1f1', 'r9i1p1f1', 'r134i1p1f1', 'r141i1p1f1', 'r146i1p1f1', 'r15i1p1f1', 'r112i1p1f1', 'r117i1p1f1', 'r125i1p1f1', 'r113i1p1f1', 'r106i1p1f1', 'r138i1p1f1', 'r5i1p1f1', 'r137i1p1f1', 'r11i1p1f1', 'r145i1p1f1', 'r114i1p1f1', 'r120i1p1f1', 'r128i1p1f1', 'r135i1p1f1', 'r110i1p1f1', 'r129i1p1f1', 'r132i1p1f1', 'r101i1p1f1', 'r124i1p1f1', 'r127i1p1f1', 'r116i1p1f1', 'r131i1p1f1', 'r121i1p1f1', 'r142i1p1f1', 'r102i1p1f1', 'r133i1p1f1', 'r111i1p1f1', 'r140i1p1f1', 'r136i1p1f1', 'r108i1p1f1', 'r130i1p1f1', 'r150i1p1f1', 'r104i1p1f1', 'r118i1p1f1', 'r109i1p1f1', 'r143i1p1f1', 'r147i1p1f1', 'r126i1p1f1', 'r13i1p1f1', 'r119i1p1f1', 'r1i1p1f1', 'r123i1p1f1', 'r122i1p1f1', 'r115i1p1f1', 'r103i1p1f1', 'r144i1p1f1', 'r139i1p1f1', 'r107i1p1f1']
Run:  r149i1p1f1
Run:  r6i1p1f1
Missing data in /climca/data/CMIP6/Scen

#### Some plots to check everything is alright

In [10]:
np.unique(merged_ts['ESM'].values)

array(['ACCESS-CM2', 'EC-Earth3', 'MPI-ESM1-2-HR', 'MRI-ESM2-0'],
      dtype='<U13')

In [11]:
merged_ts

<xarray.Dataset> Size: 498MB
Dimensions:   (ESM_run: 71, time: 3653, lat: 10, lon: 12)
Coordinates:
  * time      (time) datetime64[ns] 29kB 2015-01-01T12:00:00 ... 2024-12-31T1...
  * lat       (lat) int64 80B 47 48 49 50 51 52 53 54 55 56
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
  * lon       (lon) int64 96B 5 6 7 8 9 10 11 12 13 14 15 16
    run       (ESM_run) <U10 3kB 'r4i1p1f1' 'r5i1p1f1' ... 'r3i1p1f1' 'r1i1p1f1'
    ESM       (ESM_run) <U13 4kB 'ACCESS-CM2' 'ACCESS-CM2' ... 'MRI-ESM2-0'
  * ESM_run   (ESM_run) <U23 7kB 'ACCESS-CM2_r4i1p1f1' ... 'MRI-ESM2-0_r1i1p1f1'
Data variables:
    sfcWind   (ESM_run, time, lat, lon) float32 124MB dask.array<chunksize=(1, 1, 10, 12), meta=np.ndarray>
    rsds      (ESM_run, time, lat, lon) float32 124MB dask.array<chunksize=(1, 1, 10, 12), meta=np.ndarray>
    tas       (ESM_run, time, lat, lon) float32 124MB dask.array<chunksize=(1, 1, 10, 12), meta=np.ndarray>
    tasmax    (ESM_run, time, lat, lon) float32 124MB dask.array<chunksize=(1, 1, 10, 12), meta=np.ndarray>
Attributes:
    regrid_method:  bilinear

In [ ]:
ds = merged_ts  # assuming you've already got it as merged_ts

# 2. Select the variable you want to plot
var = "tas"
da = ds[var]

# 3. Spatially average over lat and lon
da_spatial = da.mean(dim=["lat", "lon"])
# Now da_spatial is (ESM_run, time)

# 4. Compute the ensemble‐mean per ESM (i.e. average over runs of the same model)
#    ESM is a coordinate on the ESM_run dimension
da_esm_mean = (
    da_spatial
    .groupby("ESM")           # groups the 71 runs by their ESM name
    .mean(dim="ESM_run")      # averages the runs within each ESM
)
# Now da_esm_mean has dims (ESM, time)

# 5. Collapse to annual means
da_annual = (
    da_esm_mean
    .groupby("time.year")     # group by calendar year
    .mean(dim="time")         # average all days within each year
)
# Now da_annual has dims (ESM, year)

# 6. Convert to a pandas DataFrame for easy plotting
df = da_annual.to_series().unstack("ESM")
# DataFrame indexed by year, columns are ESM names

In [ ]:
# 7. Plot
plt.figure(figsize=(10, 6))
for esm in df.columns:
    plt.plot(df.index, df[esm], label=esm)
plt.xlabel("Year")
plt.ylabel(f"Annual mean {var}")
plt.title(f"Annual mean {var} by ESM (2015–2024)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# for each ESM, first average out space and runs, then do the day-of-year grouping
all_dfs = []
for esm in np.unique(merged_ts.ESM.values):
    esm_ds = merged_ts.where(merged_ts.ESM == esm, drop=True)
    # collapse runs + space
    reduced = esm_ds.mean(dim=['ESM_run','lat','lon'])
    # group by calendar day-of-year and then mean over all years
    mean_cycle = reduced.groupby('time.dayofyear').mean(dim='time')
    
    # convert straight to DataFrame
    df = mean_cycle.to_dataframe().reset_index()
    # rename xarray’s dayofyear → day_of_year
    df = df.rename(columns={'dayofyear':'day_of_year'})
    df['ESM'] = esm
    all_dfs.append(df)

final_df = pd.concat(all_dfs, ignore_index=True)


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fcf66141940>>
Traceback (most recent call last):
  File "/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
# Plot the data
plt.figure(figsize=(12, 6))
for esm, group in final_df.groupby('ESM'):
    plt.plot(group['day_of_winter'], group['rsds'], label=esm, linewidth=2)

# Add title and labels
plt.title('Mean rsds by ESM', fontsize=16)
plt.xlabel('Day of Winter', fontsize=14)
plt.ylabel('Temperature (°C)', fontsize=14)

# Customize legend
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12, title='ESM', title_fontsize=14)

# Add grid for better readability
plt.grid(visible=True, linestyle='--', alpha=0.7)

# Adjust layout to prevent overlap
plt.tight_layout()

